# Baseline agents

## RAG Memory

In [ ]:
import os
import numpy as np
from typing import List, Dict, Any, Sequence
from student.agent.agent_rag import RAGAgent, RAGMemory
from student.agent.memory import Memory, MemoryNode

import pandas as pd
import json
import uuid
import numpy as np
from mllm import Chat, get_embeddingsx
from typing import Dict, List, Set, Iterable, Tuple
import copy

In [136]:
class MemoryNodeRAG(MemoryNode):
    id : str
    keys : Set[str]
    embeddings : List[str]
    content : str    

    def __init__(self, input: str=""):
        super().__init__(content="", keys = [input])
    
    def _get_embedding_score(self, query : str, sensitivity=0.4):
        similarity = self._get_cosine_similarity([query], keys=None)
        similarity = similarity * (similarity > sensitivity) # filter out bad matches
        return similarity 

    def get_score(self, query : str, sensitivity=0.4):
        score = self._get_embedding_score(query, sensitivity)
        return score[0][0]

    def __str__(self):
        return next(iter(self.keys))
    
    def __copy__(self):
        return copy.deepcopy(self)
    

    @classmethod
    def from_dict(cls, d):
        new_node = MemoryNodeRAG()  
        new_node._from_dict(d)
        return new_node

    def render_html(self) -> str:
        import html

        content_html = html.escape(next(iter(self.keys))).replace("\n", "<br>")

        return f"""\
    <style>
    .memory-node {{
        font-family: system-ui, sans-serif;
        border: 1px solid #ccc;
        border-radius: 8px;
        padding: 1rem;
        margin: .5rem 0;
        max-width: 200px;
    }}
    .memory-node h3 {{ margin: 0 0 .5rem 0; font-size: 1.25rem; }}
    .memory-node ul {{ margin: .25rem 0 .75rem 1rem; }}
    .memory-node p {{ margin: 0; white-space: pre-wrap; }}
    </style>

    <div class="memory-node">
    <p>{content_html}</p>
    </div>
    """


In [216]:
class MemoryRAG(Memory):
    memory : Dict[str, MemoryNodeRAG]
    keywords : Set[str]

    def __init__(self):
        super().__init__()
        self.score_matching = False
    
    def add_from_dict(self, node_dict: Dict) -> None:
        node = MemoryNodeRAG()
        node._from_dict(node_dict)
        self.add(node)

    
    def _recall(self, query: str, max_recall=5, sensitivity=0.3, thres=0.3) -> Dict[str, float]:
        nodes = self.get_nodes()
        if len(nodes) == 0:
            return {}
        
        excited_nodes = {}
        scores = self.get_scores(nodes, query, sensitivity)
        
        top_k_indices = np.argsort(-scores)
        
        for i in range(min(max_recall, len(top_k_indices))):
            s = scores[top_k_indices[i]]
            if s <= thres:
                break

            node : MemoryNodeRAG = nodes[top_k_indices[i]]
            excited_nodes[node.id] = s
        return excited_nodes
    

    def recall(self, query: str, max_recall=5, sensitivity=0.3, thres=0.3) -> Dict[str, str]:
        excited_nodes = self._recall(query, max_recall=max_recall, sensitivity=sensitivity, thres=thres)
        out = {}
        for id, score in excited_nodes.items():
            node = self.get_node(id)
            out[id] = node.__str__() 
        return out

    def load_text(self, load_path):
        # load the memory from a file
        with open(load_path) as f:
            memory_list = json.load(f)
        for d in memory_list:
            try:
                node = MemoryNodeRAG().from_dict(d)
                self.memory[node.id] = node
            except ValueError as e:
                print(e, "for dictionary: ", d)
        self.load_keywords()

    def load(self, load_path: str, clear=False) -> None:
        df = pd.read_parquet(load_path)
        if clear is True:
            self.memory.clear()
        for _, row in df.iterrows():
            node = MemoryNodeRAG.from_dict(row.to_dict())
            self.memory[node.id] = node

        self.load_keywords()

    def update_keywords(self, keyword):
        pass
    
    def load_keywords(self):
        """
        Load keywords from the memory nodes into the keyword dictionary.
        This is useful after loading a memory from a file.
        """
        self.keywords = {}

    def load_from_memory(self, memory: Memory):
        if type(memory) == MemoryRAG:
            self.memory = memory.__copy__().memory
        else:
            for id, node in memory.memory.items():
                if len(node.content) > 0:
                    new_node = MemoryNodeRAG(input=node.content)
                    new_node.id = id
                    self.add(new_node)
            print("Memory converted to RAG memory")

    def __copy__(self):
        return copy.deepcopy(self)

In [217]:
mem_rag = MemoryRAG()

#### Testing

In [218]:
mem = Memory()

In [219]:
mem.load("/Users/henrikseng/MScThesis/StudentAgent/benchmark_raspa/checkpoints/b1_matching/2/memory.parquet")

In [220]:
i = 0
for id, node in list(mem.memory.items()):
    i += 1
    if i <= 5:
        continue
    del mem.memory[id]

In [221]:
mem_rag.load_from_memory(mem)

Memory converted to RAG memory


In [59]:
q = "How to use RASPA for adsorption?"
mem_rag.recall(q, thres=0.1, sensitivity=0.1)

{'df095431': "RASPA is a classical molecular simulation software that specializes in simulations of porous systems and their interactions with liquids or gases. It's a comprehensive tool for computational chemistry and materials science, particularly valuable for studying gas storage, separation, and catalysis in frameworks like MOFs (Metal-Organic Frameworks) and zeolites. RASPA allows for different kinds of simulations for various purposes, making it a versatile platform for materials science applications involving porous materials.",
 'e568fac5': "RASPA's applications are specifically focused on porous systems and their interactions with liquids or gases. Key applications include: gas storage studies in porous materials, separation processes analysis, catalysis research in framework materials, and comprehensive materials science applications. The software is particularly valuable for studying MOFs (Metal-Organic Frameworks) and zeolites, which are important classes of porous materia

In [60]:
self = mem_rag
nodes = self.get_nodes()

excited_nodes = {}
scores = self.get_scores(nodes, q, sensitivity=0.1)

In [61]:
scores

array([0.6802667 , 0.6709459 , 0.52333015, 0.5919827 , 0.49180463],
      dtype=float32)

## RAG Agent

In [ ]:
from student.agent.agent import Agent
from student.agent.agent_memory import MemoryAgent
from student.agent.agent_memory import Ask
from student.agent.tools.tools_memory import RecallMemory
from student.agent.utils import question as q

In [132]:
class RAGRecallMemory(RecallMemory):
    def __init__(self, memory: MemoryRAG):
        super().__init__(memory)
        self.name = "recall"
        self.description = """
        Recall knowledge from your memory.
        ALWAYS choose a short description of the question as query.
        The retrieval is based on the semantic similarity of the query to the memory entries.
        """
    
    def run(self, query: str) -> Dict[str, str]:
        return super().run(query)

In [241]:
def memory_agent_tools(provider, memory_path):
    agent = RAGAgent(provider=provider, memory_path=memory_path)
    return {'agent': agent, 'tools' : [Ask(agent)]}


class AgenticRAGAgent(MemoryAgent):
    '''
    Memory Agent that allows agentic RAG for memory recall and question answering.
    '''
    
    def __init__(self, memory:Memory=None, memory_path=None, tools: dict = {}, cache=None, expensive=None, provider="openai"):

        self.memory_path = memory_path
        super().__init__(tools=tools, cache=cache, expensive=expensive, version="v1", provider=provider)

        self.add_memory(memory)

    def setup_general_prompt(self, version):
        prompt = self.get_prompt(type="general_rag", version=version)
        self.reset_system_prompt(prompt, append=True)

    def add_memory_tools(self):
        self.memory = MemoryRAG()
        recall = RAGRecallMemory(self.memory)
        self.tools[recall.name] = recall
        
        self.load_memory(self.memory_path)
    
        
    def load_memory(self, file=None):
        if file is None:
            self.add_memory(MemoryRAG())
            #print("Initializing empty RAG memory")
            return
        else:
            self.memory = Memory()
            super().load_memory(file)
            
            mem_rag = MemoryRAG()
            mem_rag.load_from_memory(self.memory)
            self.add_memory(mem_rag)
            #print("Loading memory")
    
    
    def add_memory(self,memory:Memory):
        if memory is None:
            return
        if type(memory) == MemoryRAG:
            self.memory = memory
        else: 
            self.memory = MemoryRAG()
            for id, node in memory.memory.items():
                if len(node.content) > 0:
                    new_node = MemoryNodeRAG(input=node.content)
                    new_node.id = id
                    self.memory.add(new_node)

        for tool in self.tools.values():
            if hasattr(tool, 'memory'):
                tool.memory = self.memory


    def ask(self, question: str) -> str:
        self.set_prompt(type="retrieval_rag", version="v1")
        
        prompt = f"Retrieve all knowledge related to this input: {q(question)}"
        res = self.run(prompt)
        return res

#### Testing

In [269]:
# For testing, take convert a part of a raspa memory

r = AgenticRAGAgent(memory_path="/Users/henrikseng/MScThesis/StudentAgent/benchmark_raspa/checkpoints/b1_matching/2/memory.parquet", provider="anthropic")

i = 0
for id in list(r.memory.memory.keys()):
    i += 1
    if i <= 3:
        continue
    del r.memory.memory[id]

Memory converted to RAG memory


In [61]:
r.ask("How do i setup a RASPA simulation for adsoption enthalpy?")

"**RASPA Adsorption Enthalpy at Infinite Dilution Simulation - Complete Setup Guide:**\n\n**THEORETICAL FOUNDATION:**\nThe enthalpy of adsorption at infinite dilution is calculated using:\nΔH = ⟨U_hg⟩ - ⟨U_h⟩ - ⟨U_g⟩ - RT\n\nwhere:\n- ⟨U_hg⟩ = average energy of guest molecule inside host-framework (key simulation output)\n- ⟨U_h⟩ = average energy of host-framework (0 for rigid frameworks)\n- ⟨U_g⟩ = average energy of single guest-molecule in gas phase (0 for simple molecules)\n- RT = enthalpy per particle of ideal bulk phase\n\n**CRITICAL PREREQUISITES:**\n1. **HeliumVoidFraction MUST be calculated first** in a separate simulation using Widom insertions of Helium on the framework\n2. Framework must be available as framework.cif file\n3. Molecule definition files must be properly configured\n\n**SIMULATION SETUP REQUIREMENTS:**\n- SimulationType: MonteCarlo\n- Uses single molecule insertion (CreateNumberOfMolecules 1)\n- Framework MUST be rigid for this approach to work correctly\n- Ext

## Naive RAG

In [260]:
class NaiveRAGAgent(AgenticRAGAgent):
    '''
    Naive RAG Agent
    '''
    def setup_general_prompt(self, version):
        prompt = self.get_prompt(type="naive_rag", version=version)
        self.reset_system_prompt(prompt, append=True)

    def add_memory_tools(self):
        self.memory = MemoryRAG()
        self.load_memory(self.memory_path)


    def _retrieve(self, query, sensitivity=0.1):
        res = self.memory.recall(query, sensitivity=sensitivity)
        mem = ""
        for id, i in res.items():
            mem += i
        if mem == "":
            mem = "<no memory found/>"
        return mem
    

    def run(self, prompt: str, max_iter: int=15):
        recalled = self._retrieve(prompt)
        
        rag_prompt = f"""<query>{prompt}</query>
        <context>
        {recalled}
        </context>
        <answer/>
        """
        res = super().run(rag_prompt, max_iter=max_iter)
        return res


    def get_output_jsonschema(self, remove_tools=[]):

        schema = {
        "type": "object",
        "properties": {
            "react": {
                "type": "array",
                "description": "A sequence of reasoning steps as discrete thoughts",
                "items": {
                    "type": "object",
                    "properties": {
                        "thought": {
                            "type": "string",
                            "description": "A reasoning step or internal reflection."
                        }
                    },
                    "required": ["thought"],
                    "additionalProperties": False
                },
            },
            "response": {
                "type": "string",
                "description": "Final response to the user. IGNORED IF a function is included in the react scheme"
            }
        },
        "required": ["react", "response"],
        "additionalProperties": False
        }
        return schema

In [252]:
nr = NaiveRAGAgent(memory_path="/Users/henrikseng/MScThesis/StudentAgent/benchmark_raspa/checkpoints/b1_matching/2/memory.parquet", provider="anthropic")
nr2 = NaiveRAGAgent(memory_path="/Users/henrikseng/MScThesis/StudentAgent/benchmark_raspa/checkpoints/b1_matching/2/memory.parquet", provider="anthropic")

# Test if both memories are copies
print(nr.memory)
print(nr2.memory)

Memory converted to RAG memory
Memory converted to RAG memory


In [ ]:
test_memory = mem_rag
print(test_memory)

nr = NaiveRAGAgent(memory=test_memory, provider="anthropic")
nr2 = NaiveRAGAgent(memory=test_memory, provider="anthropic")

# Test if both memories are the same
print(nr.memory)
print(nr2.memory)

In [262]:
nr.run("How to use RASPA for adsorption?")
nr.render_chat_html()